In [2]:
import torch.nn as nn
from girth import grm_mml
from torch.utils.data import dataset, DataLoader


In [3]:
import torch
from torch import nn
from torch.nn import functional as F


class VSOSPred(nn.Module):
    def __init__(self, num_usrs: int, num_tasks: int):
        super().__init__()
        self.global_strong = nn.Embedding(num_usrs, 1)
        self.performance = nn.Embedding(num_usrs * num_tasks, 1)
        nn.init.zeros_(self.global_strong.weight)
        nn.init.zeros_(self.performance.weight)
        self.num_tasks = num_tasks
    def get_strong(self, usr_ids, task_ids):
        global_strength = self.global_strong(usr_ids).squeeze(-1)
        joint_ids = usr_ids * self.num_tasks + task_ids
        local = self.performance(joint_ids).squeeze(-1)
        return global_strength + local
    def forward(self, usr_a_ids, usr_b_ids, task_ids):
        strength_a = self.get_strong(usr_a_ids, task_ids)
        strength_b = self.get_strong(usr_b_ids, task_ids)
        return {"logits": strength_a - strength_b, "strength_a": strength_a, "strength_b": strength_b}
    def regularization_loss(self, global_weight, offset_weight):
        global_penalty = self.global_strong.weight.pow(2).mean()
        offset_penalty = self.performance.weight.pow(2).mean()
        return (global_weight * global_penalty + offset_weight * offset_penalty)

In [4]:
from torch.utils.data import Dataset
import numpy as np
import pandas as pd
from dataset import VSOSet
df = pd.read_csv("learn_data.csv")
l = df['name'].values.tolist()
m = {b: a
     for a, b in enumerate(l)}
df.name = df.name.map(m)
df = df.drop(columns=['Unnamed: 0'])
dataset=VSOSet(df)
dataloader = DataLoader(dataset, shuffle=True, batch_size=64)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 475 entries, 0 to 474
Data columns (total 46 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           475 non-null    int64  
 1   total1         16 non-null     float64
 2   result1        16 non-null     float64
 3   result2        15 non-null     float64
 4   total3         198 non-null    float64
 5   grade2         198 non-null    float64
 6   stage          198 non-null    float64
 7   A              198 non-null    float64
 8   B              198 non-null    float64
 9   C              198 non-null    float64
 10  D              198 non-null    float64
 11  E              198 non-null    float64
 12  F              198 non-null    float64
 13  result3        8 non-null      float64
 14  grade3         8 non-null      float64
 15  score4         98 non-null     float64
 16  grade4         284 non-null    float64
 17  A1_x           284 non-null    float64
 18  B1_x           284 no

In [6]:
import pandas as pd
df = pd.read_csv('learn_data.csv')
df.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
name = df.name.values
df.drop(columns=['name'], inplace=True)
#df = df.drop(columns=[''])
df

,id,total1,result1,result2,total3,grade2,stage,A,B,C,...,D1_y,A2_y,B2_y,C2_y,D2_y,Sum,place,id_y,title,grade
0,0,NaN,NaN,NaN,286.0,10.0,2.0,100.0,43.0,0.0,...,9.0,100.0,65.5,38.0,12.5,388.5,NaN,NaN,NaN,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,47.0,30.0,569.0,NaN,NaN,NaN,NaN
2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,68.0,10.0,531.0,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18.0,100.0,100.0,50.0,0.0,568.0,NaN,NaN,NaN,NaN
4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,39.0,10.0,521.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,61.0,0.0,561.0,NaN,NaN,NaN,NaN
471,471,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,39.0,0.0,486.0,NaN,NaN,NaN,NaN
472,472,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,0.0,0.0,0.0,230.0,NaN,NaN,NaN,NaN
473,473,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.0,100.0,47.0,100.0,719.0,NaN,NaN,NaN,NaN


In [19]:
tar = pd.read_csv('tar1.csv')
tar

,place,name,class_study,p1,p2,p3,p4,p5,p6,p7,p8,score
0,409,Абдулгужин А.,10,100,20,20,0,51,85,23,15,314
1,403,Аболяев В.,9,100,20,20,0,73,40,48,15,316
2,449,Абрамов А.,10,100,31,0,3,80,10,35,6,265
3,191,Абубакиров Б.,11,100,36,20,31,100,85,48,25,445
4,367,Абубакиров Д.,9,100,9,20,31,100,45,32,6,343
...,...,...,...,...,...,...,...,...,...,...,...,...
475,180,Юркшус Е.,10,100,46,20,0,100,100,59,25,450
476,440,Юрчук П.,9,100,46,19,21,11,85,0,0,282
477,486,Яковлева П.,9,42,0,0,0,0,0,0,0,42
478,17,Янковский А.,10,100,31,73,37,100,85,77,67,570


In [22]:
from sklearn.metrics import ndcg_score
model = VSOSPred(num_usrs=480, num_tasks=44)
optim = torch.optim.Adam(model.parameters(), lr=1e-4)

places = tar["place"].to_numpy(dtype=np.float64)

relevance = places.max() - places
for epoch in range(50):
    model.train()
    epoch_loss = 0.0
    for batch in dataloader:
        optim.zero_grad()
        out = model(
            batch["athlete_a"],
            batch["athlete_b"],
            batch["task_id"],
        )
        pair_loss = F.binary_cross_entropy_with_logits(
            out["logits"],
            batch["target"].float(),
        )
        loss = pair_loss + model.regularization_loss(1e-3, 1e-1)
        loss.backward()
        optim.step()
        epoch_loss += loss.item()
    model.eval()
    with torch.no_grad():
        strength = (model.global_strong.weight.squeeze(-1).cpu().numpy())
        ndcg = ndcg_score(
        relevance[None, :],
        strength[None, :], k=10
        )
    mean_loss = epoch_loss / len(dataloader)
    print(
        f"Epoch {epoch + 1} | "
        f"Loss {mean_loss} | "
        f"NDCG {ndcg}"
    )

Epoch 01 | Loss 0.60742 | NDCG 0.55351
Epoch 02 | Loss 0.49410 | NDCG 0.57588
Epoch 03 | Loss 0.42923 | NDCG 0.57276
Epoch 04 | Loss 0.38584 | NDCG 0.56991
Epoch 05 | Loss 0.35430 | NDCG 0.57084
Epoch 06 | Loss 0.33061 | NDCG 0.55889
Epoch 07 | Loss 0.31256 | NDCG 0.54878
Epoch 08 | Loss 0.29871 | NDCG 0.54878
Epoch 09 | Loss 0.28801 | NDCG 0.54346
Epoch 10 | Loss 0.27970 | NDCG 0.54897
Epoch 11 | Loss 0.27321 | NDCG 0.54687
Epoch 12 | Loss 0.26811 | NDCG 0.55167
Epoch 13 | Loss 0.26406 | NDCG 0.54755
Epoch 14 | Loss 0.26084 | NDCG 0.54206
Epoch 15 | Loss 0.25826 | NDCG 0.54229
Epoch 16 | Loss 0.25617 | NDCG 0.54229
Epoch 17 | Loss 0.25447 | NDCG 0.56145
Epoch 18 | Loss 0.25308 | NDCG 0.56145
Epoch 19 | Loss 0.25193 | NDCG 0.56145
Epoch 20 | Loss 0.25098 | NDCG 0.56145
Epoch 21 | Loss 0.25018 | NDCG 0.56145
Epoch 22 | Loss 0.24951 | NDCG 0.56100
Epoch 23 | Loss 0.24895 | NDCG 0.56471
Epoch 24 | Loss 0.24848 | NDCG 0.56471
Epoch 25 | Loss 0.24807 | NDCG 0.56471
Epoch 26 | Loss 0.24772 |